# Scene-graph demo — place → move → place → hover → connect

This notebook walks through the framework end-to-end **without an LLM**, driving the same operands the Executor would pick. The point is to show:

1. The **scene graph** (deterministic, JSON-backed) updating after every operand call.
2. **Inter-operation scanning** — the bbox of the selected shape gets re-detected from CV after geometry changes, and reconciled back into the matching scene-graph object.
3. The new shape-manipulation operands: `move_shape`, `hover_object`, `connect_shapes`.

## Prerequisites

- draw.io running in a window where pyautogui can drive it (focus it during the countdown).
- On macOS, *Screen Recording* permission for the terminal running Jupyter.
- The current `state/ui_graph.json` calibrated for your drawio window.

In [1]:
# --- 0. Setup --------------------------------------------------------
import os, sys, time, json
sys.path.insert(0, os.path.abspath('..'))

from core import config
from core.tools import dispatch, TOOL_CATALOG
from core.state import scene_graph as sg

print(f'Loaded {len(TOOL_CATALOG)} operands.')
for tool_name in TOOL_CATALOG:
    print(f'  - {tool_name}')

Loaded 31 operands.
  - mouse_move
  - mouse_click
  - mouse_drag
  - key_press
  - key_combo
  - keyboard_type
  - click_empty_canvas
  - click_node
  - connect_shapes
  - double_click_node
  - drag_node
  - drag_node_near
  - extend_shape
  - hotkey
  - hover_object
  - move_shape
  - place_shape
  - press_delete
  - press_enter
  - press_escape
  - resize_node
  - resize_shape
  - rotate_shape
  - scan_handles
  - select_all
  - type_label
  - undo
  - delete_node
  - edit_label
  - move_and_deselect
  - place_and_label


In [2]:
# --- 1. Reset scene graph + share ui_graph across the notebook -------
sg.reset()

# One ui_graph dict, threaded through every dispatch() call so the
# scene_graph and selected_handles persist across cells.
g = config.ui_graph()
g['scene_graph'] = sg.load()

def show_scene():
    print(sg.summary_for_prompt(g['scene_graph']))

show_scene()

_Canvas is empty._


In [5]:
# --- 2. Focus drawio + clean canvas ---------------------------------
# Switch to your draw.io window now. After the countdown the cells
# below will start driving pyautogui clicks against whatever has focus.
# The clean phase deselects + undoes ~40 ops to clear any leftover
# shapes from prior runs so each cell starts from a known state.
#
# IMPORTANT: re-run this cell whenever you want to restart the demo —
# without it the click on the sidebar can land while a prior shape is
# selected, and drawio interprets the click against that context
# (which has caused 'wrong shape gets placed' bugs in the past).
import pyautogui
print('Switch to draw.io NOW.')
for i in range(5, 0, -1):
    print(f'  {i}s …')
    time.sleep(1)
print('GO — cleaning canvas …')
pyautogui.click(1100, 800); time.sleep(0.3)   # click empty canvas to deselect
for _ in range(40):
    pyautogui.hotkey('command', 'z')
    time.sleep(0.04)
pyautogui.click(1100, 800); time.sleep(0.3)
print('clean.')


Switch to draw.io NOW.
  5s …
  4s …
  3s …
  2s …
  1s …
GO — cleaning canvas …
clean.


In [6]:
# --- 3. Place rectangle A and label it 'Source' ----------------------
r = dispatch('place_shape', {'tool_name': 'Rectangle_Tool'}, ui_graph=g)
print('place_shape:', r)
time.sleep(0.3)
r = dispatch('type_label', {'text': 'Source'}, ui_graph=g)
print('type_label :', r)
r = dispatch('press_escape', {}, ui_graph=g)
print('press_escape:', r)

print('\n--- scene_graph after place A ---')
show_scene()

  [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter
place_shape: {'status': 'ok', 'tool': 'place_shape', 'tool_name': 'Rectangle_Tool', 'x': 36, 'y': 322, 'scene_object_id': 'obj_001', 'level': 1}
  [L1] type_label('Source')
type_label : {'status': 'ok', 'tool': 'type_label', 'text': 'Source', 'level': 1}
  [L1] press_escape
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
press_escape: {'status': 'ok', 'tool': 'press_escape', 'level': 1}

--- scene_graph after place A ---
**Objects (1):**
  - `obj_001` Rectangle "Source"  bbox=[691,504,120x60]  *SELECTED*

_(scene_graph op #3, last op: press_escape)_


In [7]:
# --- 4. Move A upward by 120 logical px ------------------------------
r = dispatch('move_shape', {'direction': 'n', 'amount': 120}, ui_graph=g)
print('move_shape:', r)
print('\n--- scene_graph after moving A north ---')
show_scene()

  [L1] move_shape('n', 120) → escape+reclick, drag (751,534) → (751,414)
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
move_shape: {'status': 'ok', 'tool': 'move_shape', 'direction': 'n', 'amount': 120, 'from': [751, 534], 'to': [751, 414], 'level': 1}

--- scene_graph after moving A north ---
**Objects (1):**
  - `obj_001` Rectangle "Source"  bbox=[691,384,120x60]  *SELECTED*

_(scene_graph op #4, last op: move_shape:n)_


In [8]:
# --- 5. Place rectangle B and label it 'Target' ----------------------
# B becomes the new selection; A is still in the scene graph at its
# moved position, and the inter-op scanner will fill in B's bbox after
# escape.
r = dispatch('place_shape', {'tool_name': 'Rectangle_Tool'}, ui_graph=g)
print('place_shape:', r)
time.sleep(0.3)
r = dispatch('type_label', {'text': 'Target'}, ui_graph=g)
print('type_label :', r)
r = dispatch('press_escape', {}, ui_graph=g)
print('press_escape:', r)

print('\n--- scene_graph after placing B ---')
show_scene()

  [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter
place_shape: {'status': 'ok', 'tool': 'place_shape', 'tool_name': 'Rectangle_Tool', 'x': 36, 'y': 322, 'scene_object_id': 'obj_002', 'level': 1}
  [L1] type_label('Target')
type_label : {'status': 'ok', 'tool': 'type_label', 'text': 'Target', 'level': 1}
  [L1] press_escape
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_a.png
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_handles_scan_b.png
press_escape: {'status': 'ok', 'tool': 'press_escape', 'level': 1}

--- scene_graph after placing B ---
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[691,384,120x60]
  - `obj_002` Rectangle "Target"  bbox=[691,504,120x60]  *SELECTED*

_(scene_graph op #7, last op: press_escape)_


In [9]:
# --- 6. Hover B without clicking ------------------------------------
# This is the 'hover without selecting' step the demo description asks
# for. drawio responds to hover by drawing the 4 directional extend
# arrows around the shape. The mouse stops on the shape; no click.

scene = g['scene_graph']
objs = scene['objects']
obj_a = next(o for o in objs if o['label'] == 'Source')
obj_b = next(o for o in objs if o['label'] == 'Target')
print(f'A = {obj_a["id"]}  bbox={obj_a["bbox"]}')
print(f'B = {obj_b["id"]}  bbox={obj_b["bbox"]}')

r = dispatch('hover_object', {'object_id': obj_b['id']}, ui_graph=g)
print('hover_object:', r)

A = obj_001  bbox=[691, 384, 120, 60]
B = obj_002  bbox=[691, 504, 120, 60]
  [L1] hover_object('obj_002') → moveTo (751,534)
hover_object: {'status': 'ok', 'tool': 'hover_object', 'object_id': 'obj_002', 'at': [751, 534], 'level': 1}


In [10]:
# --- 7. Connect B → A by dragging from B's edge anchor to A ---------
# connect_shapes computes the source anchor automatically: it picks the
# B-side that faces A (here, A is north of B so source_anchor='n').
r = dispatch(
    'connect_shapes',
    {'source_id': obj_b['id'], 'target_id': obj_a['id'], 'source_anchor': 'auto'},
    ui_graph=g,
)
print('connect_shapes:', r)

print('\n--- scene_graph after connect B → A ---')
show_scene()

  [L1] connect_shapes(obj_002→obj_001) drag (751,492) → (751,414)
connect_shapes: {'status': 'ok', 'tool': 'connect_shapes', 'source': 'obj_002', 'target': 'obj_001', 'source_anchor': 'n', 'target_anchor': 's', 'from': [751, 492], 'to': [751, 414], 'level': 1}

--- scene_graph after connect B → A ---
**Objects (2):**
  - `obj_001` Rectangle "Source"  bbox=[691,384,120x60]
  - `obj_002` Rectangle "Target"  bbox=[691,504,120x60]  *SELECTED*

**Edges (1):**
  - `edge_001` `obj_002`.n → `obj_001`.s

_(scene_graph op #8, last op: connect_shapes)_


In [11]:
# --- 8. Final state ---------------------------------------------------
# Persisted to state/scene_graph.json — the framework reads this back
# on next session, and the Executor includes it in the prompt under
# the 'SCENE GRAPH' heading.
print('Persisted state/scene_graph.json:')
with open(os.path.join(config.raw().get('paths', {}).get('state_dir', 'state'), 'scene_graph.json')) as f:
    print(json.dumps(json.load(f), indent=2))

Persisted state/scene_graph.json:
{
  "version": 1,
  "next_object_id": 3,
  "next_edge_id": 2,
  "objects": [
    {
      "id": "obj_001",
      "type": "Rectangle",
      "label": "Source",
      "bbox": [
        691,
        384,
        120,
        60
      ],
      "anchors": {
        "n": [
          751,
          384
        ],
        "s": [
          751,
          444
        ],
        "e": [
          811,
          414
        ],
        "w": [
          691,
          414
        ]
      },
      "selected": false,
      "created_op": 1,
      "last_updated_op": 4
    },
    {
      "id": "obj_002",
      "type": "Rectangle",
      "label": "Target",
      "bbox": [
        691,
        504,
        120,
        60
      ],
      "anchors": {
        "n": [
          751,
          504
        ],
        "s": [
          751,
          564
        ],
        "e": [
          811,
          534
        ],
        "w": [
          691,
          534
        ]
      },
 

## What just happened

Every operand call went through `dispatch()`, which routes to a small Python function in `core/tools/primitives.py`. None of these decisions — *which* handle to grab, *where* to drop, *which* scene-graph object to update — were made by an LLM. The framework owns them deterministically, using:

1. **`core/perception/handles.py`** — CV detection of the cyan resize dots, dark extend arrows, and rotation icon around the selected shape.
2. **`core/state/scene_graph.py`** — the typed in-memory + on-disk model: objects (id, type, label, bbox, anchors) and edges (source, target, anchor pair).
3. **`_scan_and_reconcile()`** in `primitives.py` — the inter-operation scanner. It runs after geometry-changing operands, re-detects the selection bbox, and updates the matching scene-graph object so the next operation reasons about an accurate canvas state.

When the Executor agent is involved, `core/agents/executor.py:build_prompt()` includes the scene-graph summary verbatim in the system prompt under the `## SCENE GRAPH` heading — the LLM sees object ids, types, labels, and bboxes, and decides *what* to do; the framework decides *how*.

---

## Now with the LLM in the driver's seat

Same task, but instead of us calling `dispatch()` step by step, we hand a natural-language goal to the Executor agent and let **it** pick the operands. The Executor's system prompt already includes:

- The **tool catalog** (every registered operand and its params).
- The **sidebar shapes** the perception pipeline calibrated.
- The **scene graph** as a numerical block (objects with ids/labels/bboxes, edges).
- The **active selection** with the semantic operations available on it.

So the LLM gets exactly the same numerical view of the canvas we just built up by hand. It should be able to plan and sequence the operands itself.

In [3]:
# --- LLM run · setup -------------------------------------------------
# Same focus + canvas clean + scene_graph reset as before, so the LLM
# starts from a blank slate.
import json as _json
from core.capture import screenshot as _screenshot
from core.agents.executor import infer

sg.reset()
g['scene_graph'] = sg.load()
g['selected_handles'] = None

print('Switch to draw.io NOW.')
for i in range(5, 0, -1):
    print(f'  {i}s …')
    time.sleep(1)
print('GO — cleaning canvas …')
import pyautogui
pyautogui.click(1100, 800); time.sleep(0.3)
for _ in range(40):
    pyautogui.hotkey('command', 'z'); time.sleep(0.04)
pyautogui.click(1100, 800); time.sleep(0.3)
print('clean. scene_graph reset. Ready for LLM.')

Switch to draw.io NOW.
  5s …
  4s …
  3s …
  2s …
  1s …
GO — cleaning canvas …
clean. scene_graph reset. Ready for LLM.


In [4]:
# --- LLM run · plan + execute ----------------------------------------
# The Executor picks the next operand each turn, looking at the current
# screenshot + the rendered prompt (which includes the scene_graph and
# the active selection). We dispatch its choice, then loop back with
# the result so it can decide the next step. Stops when the model
# emits `task_complete` (or we hit the budget).

TASK = (
    "Place two rectangles on the canvas. Label the first 'Source' and "
    "the second 'Target'. Then draw a connector edge from Source to "
    "Target. Use one tool per step. When the canvas matches this "
    "description, emit task_complete."
)

MAX_STEPS = 12
history = []

for step in range(1, MAX_STEPS + 1):
    print(f'\n========== LLM step {step} ==========')
    img = _screenshot(f'_llm_step{step}.png')
    decision = infer(TASK, g, img, history if history else None)
    tool = decision.get('tool')
    params = decision.get('params', {}) or {}
    reasoning = decision.get('reasoning', '')
    print(f'reasoning: {reasoning}')
    print(f'tool:      {tool}({params})')

    if tool == 'task_complete':
        print('🏁 LLM signalled task_complete')
        break
    if tool == 'request_rescan':
        print('… rescan requested; refreshing handles')
        dispatch('scan_handles', {}, ui_graph=g)
        history.append({'role':'assistant','content':_json.dumps(decision)})
        history.append({'role':'user','content':'rescanned, what next?'})
        continue
    if tool not in TOOL_CATALOG:
        print(f'❌ unknown tool {tool}; aborting')
        break

    result = dispatch(tool, params, ui_graph=g)
    print(f'result:    status={result.get("status")}'
          + (f', error={result.get("error")}' if result.get('status') != 'ok' else ''))

    history.append({'role':'assistant','content':_json.dumps({
        'tool': tool, 'params': params, 'reasoning': reasoning,
    })})
    history.append({'role':'user','content':
        f"Tool '{tool}' executed (status={result.get('status')})."
        f" What's the next step? Use 'task_complete' if the goal is met."
    })

    print('--- scene_graph after this step ---')
    show_scene()
    time.sleep(0.3)

print('\n=========== FINAL scene_graph ===========')
show_scene()


========== LLM step 1 ==========
[CAPTURE] Screenshot → /Users/zhuyuezx/Documents/UCSD/Spring_2026/CSE252D/drawioDemo/screenshots/_llm_step1.png
[EXECUTOR] Querying qwen3.5:35b …
[EXECUTOR] Raw response:
{
  "reasoning": "SCENE GRAPH shows an empty canvas. The task requires placing two rectangles. The first step is to place the first rectangle. I will use place_shape with Rectangle_Tool.",
  "tool": "place_shape",
  "params": {
    "tool_name": "Rectangle_Tool"
  }
}
[EXECUTOR] Decided: place_shape  {'tool_name': 'Rectangle_Tool'}
reasoning: SCENE GRAPH shows an empty canvas. The task requires placing two rectangles. The first step is to place the first rectangle. I will use place_shape with Rectangle_Tool.
tool:      place_shape({'tool_name': 'Rectangle_Tool'})
  [L1] place_shape('Rectangle_Tool') → click (36, 322) + Enter
result:    status=ok
--- scene_graph after this step ---
**Objects (1):**
  - `obj_001` Rectangle  bbox=?  *SELECTED*

_(scene_graph op #1, last op: place_shape)_


### Compare the two runs

If the deterministic walkthrough above ended with `obj_001 Source` and `obj_002 Target` linked by `edge_001`, the LLM run should reach an equivalent scene graph — same shapes, same connector — without us ever telling it which handle to click or which coordinates to use.

Failure modes worth watching for in the LLM run:

- **Forgets to escape** before next step → next `place_shape` happens while still in text-edit mode. The scene_graph block in the prompt would still show the most recently placed object as the selection, but its bbox is `None` because no `press_escape` ran to trigger the inter-op scan.
- **Tries `connect_shapes` before both objects have bboxes** → the operand returns an error and the LLM should recover by calling `press_escape` or `scan_handles` first.
- **Picks `extend_shape` instead of `connect_shapes`** → extend creates a brand new connected rectangle rather than linking to the existing `Target`. The scene graph will show an extra object.

Each of these is observable in the printed scene_graph after each step, which is the point of having a deterministic numerical state for the LLM to read.